<a href="https://colab.research.google.com/github/DanilRodenko/NLP-Job-Screener/blob/main/notebooks/05_Evaluation_ipynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from transformers import AutoTokenizer, DistilBertForSequenceClassification

tokenizer = AutoTokenizer.from_pretrained('/content/drive/MyDrive/NLP-Job-Screener/models/distilbert_classifier')
model = DistilBertForSequenceClassification.from_pretrained('/content/drive/MyDrive/NLP-Job-Screener/models/distilbert_classifier')

In [ ]:
import pandas as pd
from torch.utils.data import Dataset


df_jobs = pd.read_parquet('/content/drive/MyDrive/NLP-Job-Screener/data/jobs_clean.parquet')

df_jobs = df_jobs[df_jobs['category'] != 'OTHER']

df_jobs['title_and_description'] = df_jobs['title'] + " " + df_jobs['description']

from sklearn.model_selection import train_test_split

X = df_jobs['title_and_description']
y = df_jobs['category']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

from sklearn.preprocessing import LabelEncoder

# Initialize LabelEncoder
label_encoder = LabelEncoder()

# Fit on y_train and transform both y_train and y_test
y_train_encoded = label_encoder.fit_transform(y_train)
y_test_encoded = label_encoder.transform(y_test)



class JobDataset(Dataset):
  def __init__(self, texts, labels, tokenizer, max_len=512):
    self.texts = texts
    self.labels = labels
    self.tokenizer = tokenizer
    self.max_len = max_len

  def __len__(self):
    return len(self.texts)

  def __getitem__(self, idx):
    text = str(self.texts[idx]) # Ensure text is string
    encoding = self.tokenizer(
    text,
    add_special_tokens=True,
    max_length=self.max_len,
    padding='max_length',
    return_attention_mask=True,
    return_tensors='pt',
    truncation=True
)

    return {
        'input_ids': encoding['input_ids'].flatten(),
        'attention_mask': encoding['attention_mask'].flatten(),
        'labels': torch.tensor(self.labels[idx], dtype=torch.long)
    }

train_dataset = JobDataset(X_train.values, y_train_encoded, tokenizer)
test_dataset = JobDataset(X_test.values, y_test_encoded, tokenizer)

from torch.utils.data import DataLoader

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=16)

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

In [ ]:
import torch
import numpy as np
from sklearn.metrics import classification_report

# Put model in evaluation mode
model.eval()

all_predictions = []
all_true_labels = []

with torch.no_grad():
    for batch in test_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        outputs = model(input_ids, attention_mask=attention_mask)
        logits = outputs.logits

        predictions = torch.argmax(logits, dim=-1)
        all_predictions.extend(predictions.cpu().numpy())
        all_true_labels.extend(labels.cpu().numpy())

# Calculate classification report
target_names = label_encoder.inverse_transform(np.arange(len(label_encoder.classes_)))
print(classification_report(all_true_labels, all_predictions, target_names=target_names))

In [ ]:
print(next(model.parameters()).device)

In [ ]:
import joblib
joblib.dump(label_encoder, '/content/drive/MyDrive/NLP-Job-Screener/models/label_encoder.joblib')